In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [3]:
df = pd.read_csv('restaurant_with_hygiene_score.csv')


In [4]:
df.head()

,BusinessName,BusinessType,RatingValue,InspectionDate,Address,PostCode,Province,user_rating,Hygiene_Score
0,Restaurant_1,Restaurant/Cafe/Shop,Pass,2024-12-17,Address_1,40100,Punjab,3.1,76.9
1,Restaurant_2,Restaurant/Cafe/Shop,Pass,2025-07-06,Address_2,46000,Punjab,1.8,65.2
2,Restaurant_3,Restaurant/Cafe/Shop,Pass,2025-06-16,Address_3,46000,Punjab,1.1,58.9
3,Restaurant_4,Restaurant/Cafe/Shop,Pass,2025-03-06,Address_4,60000,Punjab,2.3,69.7
4,Restaurant_5,Restaurant/Cafe/Shop,Pass,2024-06-10,Address_5,60000,Punjab,2.3,69.7


In [5]:
df = df.dropna()

# Convert date
df['InspectionDate'] = pd.to_datetime(df['InspectionDate'])

In [6]:
scaler = MinMaxScaler()

df[['Hygiene_Score', 'user_rating']] = scaler.fit_transform(
    df[['Hygiene_Score', 'user_rating']]
)

In [7]:
df['final_score'] = 0.7 * df['Hygiene_Score'] + 0.3 * df['user_rating']

In [8]:
top_restaurants = df.sort_values(by='final_score', ascending=False)

print(top_restaurants[['BusinessName', 'final_score']].head(10))

       BusinessName  final_score
379  Restaurant_380     1.000000
354  Restaurant_355     1.000000
89    Restaurant_90     1.000000
154  Restaurant_155     1.000000
201  Restaurant_202     1.000000
259  Restaurant_260     1.000000
34    Restaurant_35     0.980724
511  Restaurant_512     0.980724
26    Restaurant_27     0.980724
295  Restaurant_296     0.961449


In [11]:
def recommend_by_location(province):
    filtered = df[df['Province'] == province]
    return filtered.sort_values(by='final_score', ascending=False).head(5)

def safe_restaurants(threshold=0.8):
    return df[df['Hygiene_Score'] > threshold].sort_values(by='final_score', ascending=False)

In [12]:
recommend_by_location("Punjab")

,BusinessName,BusinessType,RatingValue,InspectionDate,Address,PostCode,Province,user_rating,Hygiene_Score,final_score
379,Restaurant_380,Restaurant/Cafe/Shop,Pass,2024-12-09,Address_380,54000,Punjab,1.0,1.0,1.0
354,Restaurant_355,Restaurant/Cafe/Shop,Pass,2025-02-26,Address_355,55150,Punjab,1.0,1.0,1.0
89,Restaurant_90,Restaurant/Cafe/Shop,Pass,2025-09-08,Address_90,54000,Punjab,1.0,1.0,1.0
154,Restaurant_155,Restaurant/Cafe/Shop,Pass,2024-05-16,Address_155,55150,Punjab,1.0,1.0,1.0
201,Restaurant_202,Restaurant/Cafe/Shop,Pass,2024-01-20,Address_202,54000,Punjab,1.0,1.0,1.0


In [13]:
safe_restaurants()

,BusinessName,BusinessType,RatingValue,InspectionDate,Address,PostCode,Province,user_rating,Hygiene_Score,final_score
89,Restaurant_90,Restaurant/Cafe/Shop,Pass,2025-09-08,Address_90,54000,Punjab,1.000,1.000000,1.000000
154,Restaurant_155,Restaurant/Cafe/Shop,Pass,2024-05-16,Address_155,55150,Punjab,1.000,1.000000,1.000000
201,Restaurant_202,Restaurant/Cafe/Shop,Pass,2024-01-20,Address_202,54000,Punjab,1.000,1.000000,1.000000
379,Restaurant_380,Restaurant/Cafe/Shop,Pass,2024-12-09,Address_380,54000,Punjab,1.000,1.000000,1.000000
354,Restaurant_355,Restaurant/Cafe/Shop,Pass,2025-02-26,Address_355,55150,Punjab,1.000,1.000000,1.000000
...,...,...,...,...,...,...,...,...,...,...
370,Restaurant_371,Restaurant/Cafe/Shop,Pass,2024-01-29,Address_371,55150,Punjab,0.725,0.814953,0.787967
432,Restaurant_433,Restaurant/Cafe/Shop,Pass,2024-07-06,Address_433,40100,Punjab,0.725,0.814953,0.787967
487,Restaurant_488,Restaurant/Cafe/Shop,Pass,2024-07-31,Address_488,40100,Punjab,0.725,0.814953,0.787967
492,Restaurant_493,Restaurant/Cafe/Shop,Pass,2025-02-18,Address_493,54000,Punjab,0.725,0.814953,0.787967


In [21]:
#Above this is recommendation logic (based on location), Below is checking multiple models
# Remove extreme hygiene values
df = df[(df['Hygiene_Score'] > df['Hygiene_Score'].quantile(0.05)) &
        (df['Hygiene_Score'] < df['Hygiene_Score'].quantile(0.95))]

X = df[['user_rating', 'Province', 'BusinessType']]
y = df['Hygiene_Score']


In [22]:
X = pd.get_dummies(X, columns=['Province', 'BusinessType'])

In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [28]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

grid = GridSearchCV(RandomForestRegressor(), param_grid, cv=3, n_jobs=-1)
grid.fit(X_train, y_train)

model = grid.best_estimator_

In [29]:
from sklearn.metrics import mean_absolute_error, r2_score

y_pred = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("R2 Score:", r2_score(y_test, y_pred))

MAE: 0.08633134350100403
R2 Score: 0.6403378927713337


In [30]:
from xgboost import XGBRegressor

model2 = XGBRegressor(n_estimators=200, learning_rate=0.1)
model2.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=200,
             n_jobs=None, num_parallel_tree=None, ...)

In [31]:
y_pred2 = model2.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred2))
print("R2 Score:", r2_score(y_test, y_pred2))

MAE: 0.08558182619794269
R2 Score: 0.6454657645907129
